# Etapa 2 · Prueba tu agente Christopher

**Autor:** Christopher. **Grupo/integrantes:** completar aquí.

Adaptación de `a2a-test.ipynb` del taller. Primero consultamos la API de LangGraph; después repetimos una consulta por A2A. El agente y sus herramientas permanecen sin cambios.

Este archivo está **preparado, todavía no ejecutado**. Las respuestas válidas para la entrega son las que obtengas al correr las celdas; guarda con **Ctrl+S y no limpies las salidas**. Ejecutar todo realizará consultas al modelo configurado en el servidor.

> **Integración Grupo 6:** en el repositorio compartido este perfil corre en el puerto **2026**
> (Rodrigo en 2025). Define `A2A_PORT=2026` antes de iniciar el kernel. El notebook de la
> Etapa 3 está en [`../../notebooks/a2a-test.ipynb`](../../notebooks/a2a-test.ipynb).

## Antes de ejecutar

1. En una terminal PowerShell, sitúate en la carpeta `mi-agente` y ejecuta:

```powershell
$env:PYTHONUTF8 = "1"
.\.venv\Scripts\langgraph.exe dev --port 2025 --no-browser
```

Deja esa terminal abierta. `PYTHONUTF8` evita problemas con caracteres de los logs en Windows. La clave debe estar en el `.env` del servidor; no se copia al notebook.

2. Abre este notebook y selecciona como kernel el intérprete **mi-agente/.venv/Scripts/python.exe**. Si no aparece, en VS Code utiliza «Seleccionar kernel → Entornos de Python → Seleccionar intérprete» y busca esa ruta.
3. Ejecuta las celdas de arriba abajo con **Shift+Enter**. Conserva la salida de descubrimiento, las tres pruebas, la tarjeta y la Task A2A.

In [1]:
import json
import os
import sys
import uuid
from pathlib import Path
import httpx
from langgraph_sdk import get_client

LANGGRAPH_URL = os.environ.get("A2A_URL", "http://127.0.0.1:2025")
A2A_HOST = "127.0.0.1"
A2A_PORT = int(os.environ.get("A2A_PORT", "2025"))
A2A_URL = os.environ.get("A2A_URL", f"http://{A2A_HOST}:{A2A_PORT}")
GRAPH_ID = "agente_christopher"
TIMEOUT = 180

# Solo verifica la ruta del kernel; no lee ni muestra claves.
carpeta_actual = Path.cwd().resolve()
PROYECTO = next((p for p in [carpeta_actual, *carpeta_actual.parents]
                 if (p / "langgraph.json").is_file() and (p / "src/agente_christopher").is_dir()), None)
if PROYECTO is None:
    raise RuntimeError("Abre el notebook dentro de mi-agente/notebooks.")
if Path(sys.prefix).resolve() != (PROYECTO / ".venv").resolve():
    raise RuntimeError(f"Selecciona el kernel {PROYECTO / '.venv/Scripts/python.exe'}. Actual: {sys.executable}")

# No enviar claves de LangSmith/Google desde el notebook al servidor local.
lg = get_client(url=LANGGRAPH_URL, api_key=None, timeout=TIMEOUT)
evidencias = {}

def mostrar(valor):
    print(json.dumps(valor, indent=2, ensure_ascii=False, default=str))

print("Kernel:", sys.executable)
print("Servidor:", LANGGRAPH_URL)
print("Grafo esperado:", GRAPH_ID)

Kernel: /Users/rodrigocaceresvaldes/curso-agentic-ai/tmp/grupo-6-taller-a2a/agente-christopher/.venv/bin/python
Servidor: http://127.0.0.1:2026
Grafo esperado: agente_christopher


## 1. Encontrar el Id de tu agente

El `graph_id` es el nombre registrado. El `assistant_id` es el UUID asignado por el servidor. Esta celda lista los asistentes y guarda la evidencia del descubrimiento.

In [2]:
try:
    asistentes = await lg.assistants.search(limit=100, response_format="array")
except httpx.ConnectError as exc:
    raise RuntimeError("No se pudo conectar. Inicia langgraph dev --port 2025 desde mi-agente y revisa su terminal.") from exc

for asistente in asistentes:
    mostrar({k: asistente.get(k) for k in ("name", "graph_id", "assistant_id")})
if not asistentes:
    raise RuntimeError("El servidor no devolvió asistentes. Revisa langgraph.json y la terminal.")

{
  "name": "agente_christopher",
  "graph_id": "agente_christopher",
  "assistant_id": "b71a76f3-1a4f-5263-a63d-b005bf9e46e1"
}


In [3]:
# Equivale a copiar el UUID correspondiente a tu grafo, sin fijar el de otra instalación.
candidatos = [a for a in asistentes if a.get("graph_id") == GRAPH_ID]
if len(candidatos) != 1:
    raise RuntimeError(f"Se esperaba un asistente para {GRAPH_ID}; encontrados: {len(candidatos)}. Revisa el listado.")
ASSISTANT_ID = candidatos[0]["assistant_id"]
uuid.UUID(ASSISTANT_ID)
evidencias["descubrimiento"] = {"graph_id": GRAPH_ID, "assistant_id": ASSISTANT_ID}
print("ASSISTANT_ID =", ASSISTANT_ID)

ASSISTANT_ID = b71a76f3-1a4f-5263-a63d-b005bf9e46e1


## 2. Comunicación directa con el agente · funciones auxiliares

`crear_entrada` construye el mensaje. `extraer_texto` recupera la respuesta final del estado, que puede incluir mensajes y resultados de herramientas. `consultar_langgraph` hace la consulta. Con `thread_id=None` es independiente; con el mismo hilo conserva contexto.

In [4]:
OUTPUT_KEY = None

def crear_entrada(texto):
    return {"messages": [{"role": "human", "content": texto}]}

def extraer_texto(estado):
    if not isinstance(estado, dict):
        raise ValueError("La salida no es un estado de grafo.")
    if estado.get("__interrupt__"):
        raise RuntimeError("El grafo quedó pausado; revisa el interrupt.")
    if OUTPUT_KEY:
        contenido = estado[OUTPUT_KEY]
    else:
        respuesta = next((m for m in reversed(estado.get("messages", []))
                          if m.get("type", m.get("role")) in ("ai", "assistant")), None)
        if respuesta is None:
            raise ValueError("No hay respuesta ai/assistant. Revisa la terminal del servidor.")
        contenido = respuesta.get("content", "")
    if isinstance(contenido, list):
        contenido = "\n".join(b if isinstance(b, str) else b.get("text", "")
                              for b in contenido if isinstance(b, (str, dict)))
    if not isinstance(contenido, str) or not contenido.strip():
        raise ValueError("El agente no devolvió texto final.")
    return contenido

def herramientas_usadas(estado):
    return [t.get("name", "") for m in estado.get("messages", [])
            for t in m.get("tool_calls", [])]

async def consultar_langgraph(texto, thread_id=None):
    estado = await lg.runs.wait(thread_id, ASSISTANT_ID, input=crear_entrada(texto))
    return extraer_texto(estado)

## 3. Pregunta 1 · Personalidad

Debe presentarse como Christopher, especialista en costos de software, con tono amable, técnico y breve. Puedes cambiar `PREGUNTA` y volver a ejecutar esta celda. Cada ejecución inicia una consulta independiente.

In [5]:
PREGUNTA = "Preséntate y cuéntame cómo me puedes ayudar."
estado_personalidad = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA))
respuesta_personalidad = extraer_texto(estado_personalidad)
print("PREGUNTA:", PREGUNTA)
print("\nRESPUESTA:\n", respuesta_personalidad)
evidencias["personalidad"] = {"pregunta": PREGUNTA, "respuesta": respuesta_personalidad}

PREGUNTA: Preséntate y cuéntame cómo me puedes ayudar.

RESPUESTA:
 ¡Hola! Soy Christopher, especialista en evaluación de costos de programación y desarrollo de software. 

Estoy aquí para ayudarte a estimar y comparar los costos de tus proyectos informáticos de forma clara y técnica, utilizando un tarifario simulado en pesos chilenos (CLP, sin IVA) basado en nuestros apuntes del taller.

Puedo ayudarte a:
- Estimar el costo total de desarrollo considerando recursos, horas y servicios recurrentes (con sus respectivos períodos).
- Comparar distintos escenarios o alternativas técnicas para tu proyecto.
- Explicar detalladamente el desglose de cada partida.

¿Tienes en mente algún proyecto o requerimiento que te gustaría que evaluemos hoy?


## 4. Pregunta 2 · Conocimientos y skills

Pedimos utilizar el caso ficticio de los apuntes para estimar. Revisa que consulte `consultar_mis_apuntes`, utilice el tarifario y calcule con `calcular_estimacion`. Debe identificar los valores como simulados y mostrar 7 u 8 partidas.

**Referencia aritmética del caso acordado:** 847.000 CLP; 820.000 de trabajo inicial y 27.000 de servicios (1 mes de alojamiento y 1 año de dominio). Es una referencia para contrastar, no una respuesta insertada como evidencia.

In [6]:
PREGUNTA_CONOCIMIENTOS = (
    "Usa el caso ficticio de tus apuntes y aplica tu habilidad de estimar costos: "
    "evalúa la aplicación web de solicitudes internas con las cantidades exactas del caso. "
    "Consulta tus apuntes y tarifas; calcula con tu herramienta. "
    "Muestra el total en CLP y las 8 partidas, distinguiendo trabajo inicial y períodos de servicios."
)
estado_conocimientos = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA_CONOCIMIENTOS))
respuesta_conocimientos = extraer_texto(estado_conocimientos)
usadas = herramientas_usadas(estado_conocimientos)
print("PREGUNTA:", PREGUNTA_CONOCIMIENTOS)
print("\nRESPUESTA:\n", respuesta_conocimientos)
print("\nHERRAMIENTAS OBSERVADAS:", usadas)
evidencias["conocimientos"] = {"pregunta": PREGUNTA_CONOCIMIENTOS, "respuesta": respuesta_conocimientos, "herramientas": usadas}
if not {"consultar_mis_apuntes", "consultar_tarifas", "calcular_estimacion"}.issubset(usadas):
    print("REVISAR: no aparecen todas las herramientas esperadas. Registra lo observado antes de ajustar el agente.")

PREGUNTA: Usa el caso ficticio de tus apuntes y aplica tu habilidad de estimar costos: evalúa la aplicación web de solicitudes internas con las cantidades exactas del caso. Consulta tus apuntes y tarifas; calcula con tu herramienta. Muestra el total en CLP y las 8 partidas, distinguiendo trabajo inicial y períodos de servicios.

RESPUESTA:
 ¡Hola! Con gusto te comparto la evaluación de costos para la aplicación web de solicitudes internas, utilizando los datos del caso ficticio y el tarifario simulado (`simulado-v1`, sin IVA).

### Total General
**847.000 CLP** *(Costo inicial: 820.000 CLP | Servicios por sus períodos: 27.000 CLP)*

### Desglose de Partidas

| Concepto | Cantidad y Unidad | Tarifa (CLP) | Subtotal (CLP) |
| :--- | :--- | :--- | :--- |
| Análisis de requerimientos | 4 hora(s) | 18.000 | 72.000 |
| Desarrollo de interfaz | 12 hora(s) | 20.000 | 240.000 |
| Desarrollo de lógica y datos | 16 hora(s) | 22.000 | 352.000 |
| Pruebas funcionales | 6 hora(s) | 16.000 | 96.000 |

## 5. Pregunta 3 · Memoria

Creamos un hilo una sola vez y hacemos dos llamadas con el mismo `thread_id`. Al repetir esta primera celda crearás otra conversación. Para seguir la conversación actual, vuelve a ejecutar solo la celda siguiente, sin crear otro hilo.

In [7]:
thread = await lg.threads.create()
thread_id = thread["thread_id"]
OBJETIVO = "Crear una aplicación web de solicitudes internas para una oficina con computadores y conectividad existentes."
MENSAJE_OBJETIVO = f"Mi objetivo para este proyecto es: {OBJETIVO} Guárdalo como contexto para esta conversación."
respuesta_objetivo = await consultar_langgraph(MENSAJE_OBJETIVO, thread_id=thread_id)
print("THREAD_ID:", thread_id)
print("MENSAJE:", MENSAJE_OBJETIVO)
print("\nRESPUESTA:\n", respuesta_objetivo)

THREAD_ID: 01a15076-43d5-7ec3-a769-b7306271a784
MENSAJE: Mi objetivo para este proyecto es: Crear una aplicación web de solicitudes internas para una oficina con computadores y conectividad existentes. Guárdalo como contexto para esta conversación.

RESPUESTA:
 ¡Hola! Qué gusto saludarte. Con mucho gusto te ayudaré a estimar los costos para tu aplicación web de solicitudes internas.

Como se trata de una oficina que ya cuenta con computadores y conectividad, no necesitamos incurrir en gastos de infraestructura base o equipamiento adicional. 

Antes de hacer los cálculos oficiales, ¿podrías darme un poco más de detalle sobre el alcance? Por ejemplo:
1. ¿Qué funcionalidades principales debería tener la aplicación (ej. creación de tickets, estados, asignación de responsables, notificaciones)?
2. ¿Cuántos usuarios estimados utilizarán la plataforma de forma concurrente?

Con esos datos podré seleccionar los recursos y horas adecuados para preparar una estimación precisa y detallada. ¡Quedo

In [8]:
PREGUNTA_MEMORIA = "¿Qué objetivo te acabo de contar?"
respuesta_memoria = await consultar_langgraph(PREGUNTA_MEMORIA, thread_id=thread_id)
print("MISMO THREAD_ID:", thread_id)
print("PREGUNTA:", PREGUNTA_MEMORIA)
print("\nRESPUESTA:\n", respuesta_memoria)
evidencias["memoria"] = {"thread_id": thread_id, "objetivo": OBJETIVO, "primera_respuesta": respuesta_objetivo, "pregunta": PREGUNTA_MEMORIA, "respuesta": respuesta_memoria}

MISMO THREAD_ID: 01a15076-43d5-7ec3-a769-b7306271a784
PREGUNTA: ¿Qué objetivo te acabo de contar?

RESPUESTA:
 Me contaste que el objetivo del proyecto es crear una **aplicación web de solicitudes internas para una oficina con computadores y conectividad existentes**. 

Ya lo tengo guardado como contexto. ¿Avanzamos con las funcionalidades y usuarios estimados para armar la estimación?


## 6. Vamos a Agent2Agent · descargar la tarjeta

Hasta ahora usamos la API de LangGraph. Ahora usamos el protocolo A2A. El resolver pide la tarjeta incluyendo el `assistant_id`; sin ese parámetro el servidor puede rechazar la petición.

La tarjeta describe capacidades y dirección del agente. Fijamos `protocol_version = "0.3.0"` en la interfaz JSONRPC porque `langgraph-api` 0.16 incluye campos `kind` que el protobuf de `a2a-sdk` 1.2.2 no reconoce al parsear la Task. Es la misma compatibilidad que usa el notebook de Etapa 3.

In [9]:
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest, TaskState
from a2a.helpers import new_text_message, get_message_text, get_artifact_text
from google.protobuf.json_format import MessageToDict

# Permite repetir esta celda sin dejar clientes abiertos.
if "cliente_a2a" in globals():
    await cliente_a2a.close()
if "http_a2a" in globals() and not http_a2a.is_closed:
    await http_a2a.aclose()

http_a2a = httpx.AsyncClient(timeout=TIMEOUT, trust_env=False)
resolver = A2ACardResolver(http_a2a, A2A_URL)
card_remota = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
for interfaz in card_remota.supported_interfaces:
    if interfaz.protocol_binding == "JSONRPC":
        # langgraph-api 0.16 incluye "kind" en la Task; el protobuf de a2a-sdk 1.2.2
        # no lo reconoce. 0.3.0 usa el formato compatible del transporte JSONRPC.
        interfaz.protocol_version = "0.3.0"

print("AGENT CARD DESCARGADA:")
mostrar(MessageToDict(card_remota))
evidencias["tarjeta"] = MessageToDict(card_remota)

cliente_a2a = await create_client(
    card_remota,
    client_config=ClientConfig(
        httpx_client=http_a2a, streaming=False,
        supported_protocol_bindings=["JSONRPC"],
    ),
)
print("Cliente A2A preparado. Mantener abierto hasta enviar el mensaje.")

AGENT CARD DESCARGADA:
{
  "name": "agente_christopher",
  "description": "agente_christopher assistant",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2026/a2a/b71a76f3-1a4f-5263-a63d-b005bf9e46e1",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "0.3.0"
    }
  ],
  "version": "0.16.1",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "params": {
          "default": "context",
          "header": "LangGraph-A2A-History-Scope",
          "methods": [
            "SendMessage",
            "GetTask",
            "ListTasks"
          ],
          "values": [
            "context",
            "task"
          ]
        }
      }
    ]
  },
  "defaultInputModes": [
    "application/json",
    "text/plain"
  ],
  "defaultOutputModes": [
    "applicat

## 7. Repetir la consulta por A2A · Message → Task → Artifact

Repetimos la consulta del caso de los apuntes, ahora por A2A. El mensaje tiene su propio identificador y un `contextId` UUID. La Task identifica el trabajo; sus artifacts contienen la salida. Mostramos el resultado estructurado completo y el texto.

Este hilo A2A es independiente de la prueba de memoria anterior. Al repetir esta celda conserva su UUID. El envío puede esperar mientras el modelo consulta las herramientas; si falla, revisa también la terminal del servidor.

In [10]:
if http_a2a.is_closed:
    raise RuntimeError("El cliente está cerrado. Reejecuta la celda de tarjeta antes de enviar.")
if "context_id_a2a" not in globals():
    context_id_a2a = str(uuid.uuid4())

PREGUNTA_A2A = PREGUNTA_CONOCIMIENTOS
mensaje_a2a = new_text_message(PREGUNTA_A2A, role=Role.ROLE_USER, context_id=context_id_a2a)
solicitud_a2a = SendMessageRequest(message=mensaje_a2a)
print("MENSAJE A2A:")
mostrar(MessageToDict(mensaje_a2a))

textos_a2a = []
tareas_a2a = []
async for evento in cliente_a2a.send_message(solicitud_a2a):
    if evento.HasField("task"):
        tarea = evento.task
        tareas_a2a.append(MessageToDict(tarea))
        print("\nTASK DEVUELTA:")
        mostrar(MessageToDict(tarea))
        print("ESTADO:", TaskState.Name(tarea.status.state))
        if tarea.status.state != TaskState.TASK_STATE_COMPLETED:
            raise RuntimeError(f"La Task {tarea.id} no terminó correctamente. Revisa su estado y la terminal.")
        textos_a2a.extend(get_artifact_text(a) for a in tarea.artifacts)
        if not any(textos_a2a) and tarea.status.HasField("message"):
            textos_a2a.append(get_message_text(tarea.status.message))
    elif evento.HasField("message"):
        print("\nRESPUESTA TIPO MESSAGE:")
        mostrar(MessageToDict(evento.message))
        textos_a2a.append(get_message_text(evento.message))

respuesta_a2a = "\n".join(t for t in textos_a2a if t.strip())
if not respuesta_a2a.strip():
    raise RuntimeError("No se recibió texto de respuesta por A2A.")
print("\nRESPUESTA A2A EN TEXTO:\n", respuesta_a2a)
evidencias["a2a"] = {"context_id": context_id_a2a, "pregunta": PREGUNTA_A2A, "respuesta": respuesta_a2a, "tasks": tareas_a2a}
if not tareas_a2a:
    print("REVISAR ENTREGA: hubo respuesta Message, pero esta ejecución no contiene la Task solicitada por el taller.")

MENSAJE A2A:
{
  "messageId": "7e1edf63-e50f-4ad0-b920-8e4f6f2ba6ce",
  "contextId": "91ca3382-a786-409b-acb4-75058bc05b2a",
  "role": "ROLE_USER",
  "parts": [
    {
      "text": "Usa el caso ficticio de tus apuntes y aplica tu habilidad de estimar costos: evalúa la aplicación web de solicitudes internas con las cantidades exactas del caso. Consulta tus apuntes y tarifas; calcula con tu herramienta. Muestra el total en CLP y las 8 partidas, distinguiendo trabajo inicial y períodos de servicios."
    }
  ]
}



TASK DEVUELTA:
{
  "id": "91ca3382-a786-409b-acb4-75058bc05b2a:01a15076-5433-7143-bd2f-f52da33aec1f",
  "contextId": "91ca3382-a786-409b-acb4-75058bc05b2a",
  "status": {
    "state": "TASK_STATE_COMPLETED",
    "timestamp": "2026-10-18T19:21:39.779089Z"
  },
  "artifacts": [
    {
      "artifactId": "c9b62a43-fc6f-4352-bbed-e6c8480898ec",
      "name": "Assistant Response",
      "description": "Response from assistant b71a76f3-1a4f-5263-a63d-b005bf9e46e1",
      "parts": [
        {
          "text": "¡Hola! Con gusto te presento la estimación para la aplicación web de solicitudes internas, basada en el caso y tarifario simulados (`simulado-v1`, sin IVA).\n\n**Total estimado:** **847.000 CLP** (Costo inicial: 820.000 CLP; Servicios recurrentes: 27.000 CLP por sus períodos).\n\n| Concepto | Cantidad y Unidad | Tarifa (CLP) | Subtotal (CLP) |\n| :--- | :--- | :--- | :--- |\n| Análisis de requerimientos | 4 hora(s) | 18.000 | 72.000 |\n| Desarrollo de interfaz | 12 hora(s) | 20.000 | 

## 8. Revisar evidencias y completar tus observaciones

Esta revisión detecta secciones ejecutadas; **no evalúa automáticamente la calidad de las respuestas**. Contrasta el tono, datos, herramientas y memoria con el perfil diseñado.

In [11]:
requeridas = ["descubrimiento", "personalidad", "conocimientos", "memoria", "tarjeta", "a2a"]
for nombre in requeridas:
    print(f"{nombre}: {'EJECUTADO' if nombre in evidencias else 'PENDIENTE'}")
if "a2a" in evidencias:
    print("Task A2A guardada:", bool(evidencias["a2a"]["tasks"]))
print("Guarda el notebook con Ctrl+S para conservar sus salidas reales.")

descubrimiento: EJECUTADO
personalidad: EJECUTADO
conocimientos: EJECUTADO
memoria: EJECUTADO
tarjeta: EJECUTADO
a2a: EJECUTADO
Task A2A guardada: True
Guarda el notebook con Ctrl+S para conservar sus salidas reales.


### Párrafo para el documento de entrega · completar después de probar

**Estado:** pendiente de ejecución y revisión por Christopher.

- **Personalidad:** ¿Se presentó con tu nombre, especialidad y tono? Escribe lo observado.
- **Conocimientos y habilidades:** ¿Consultó tus apuntes y herramientas? ¿Mostró 7 u 8 partidas y el costo esperado? Explica cualquier diferencia.
- **Memoria:** ¿Recordó el objetivo usando el mismo hilo?
- **A2A:** ¿Se descargó la tarjeta y volvió una Task completada con su resultado?
- **Ajustes:** indica qué cambiaste realmente en el prompt o apuntes después de probar. Si no cambiaste nada, explica por qué. Si necesitas ajustar el agente, vuelve a la etapa 1, reinicia el servidor y repite las pruebas afectadas.

Redacta aquí el párrafo final con tus resultados. No presentes estas instrucciones como conclusiones ni des por exitosa una prueba que no corriste.

**Entregables:** notebook ejecutado con salidas guardadas; capturas o copias de respuestas en el documento de entrega; párrafo de análisis y ajustes. La selección de varios agentes y su conversación se desarrollarán en la etapa 3.

## Preparar mi tarjeta para la Etapa 3

En la integración del grupo, Christopher corre en el puerto **2026** y Rodrigo en el **2025**.
Si ejecutas este notebook contra el servidor de 2026, define `A2A_PORT=2026` (o `A2A_URL`) antes
de iniciar el kernel. La celda actualiza el nombre y la descripción que el supervisor leerá en la
Agent Card; es idempotente.

In [12]:
# Prepara la tarjeta de Christopher para la Etapa 3.
# El servidor A2A toma el nombre y la descripción del asistente para armar la Agent Card.
# Esta celda es idempotente: puedes ejecutarla más de una vez.
NOMBRE_TARJETA = "Christopher"
DESCRIPCION_TARJETA = (
    "Especialista en evaluación de costos de programación y desarrollo de software. "
    "Estima, compara y explica costos en CLP con un tarifario simulado fijo, muestra "
    "7 u 8 partidas y separa el trabajo inicial de los servicios recurrentes. "
    "No compra, contrata ni despliega; pide contexto cuando falta alcance."
)

await lg.assistants.update(ASSISTANT_ID, name=NOMBRE_TARJETA, description=DESCRIPCION_TARJETA)
asistente_actualizado = await lg.assistants.get(ASSISTANT_ID)
print("Tarjeta preparada:")
print(json.dumps({
    "name": asistente_actualizado.get("name"),
    "description": asistente_actualizado.get("description"),
    "graph_id": asistente_actualizado.get("graph_id"),
    "assistant_id": asistente_actualizado.get("assistant_id"),
}, indent=2, ensure_ascii=False))


Tarjeta preparada:
{
  "name": "Christopher",
  "description": "Especialista en evaluación de costos de programación y desarrollo de software. Estima, compara y explica costos en CLP con un tarifario simulado fijo, muestra 7 u 8 partidas y separa el trabajo inicial de los servicios recurrentes. No compra, contrata ni despliega; pide contexto cuando falta alcance.",
  "graph_id": "agente_christopher",
  "assistant_id": "b71a76f3-1a4f-5263-a63d-b005bf9e46e1"
}


## Etapa 3 · Conversación con Rodrigo

La conversación entre los dos especialistas vive en el notebook compartido de la raíz:
[`notebooks/a2a-test.ipynb`](../../notebooks/a2a-test.ipynb).

## 9. Cerrar conexiones · ejecutar al terminar

No borra respuestas ni conversaciones del servidor. Si luego quieres repetir el envío A2A, ejecuta de nuevo la celda de tarjeta y cliente.

In [13]:
if "cliente_a2a" in globals():
    await cliente_a2a.close()
if "http_a2a" in globals() and not http_a2a.is_closed:
    await http_a2a.aclose()
print("Conexiones A2A cerradas. Guarda el notebook sin limpiar salidas.")

Conexiones A2A cerradas. Guarda el notebook sin limpiar salidas.


## Si aparece un error

- **No conecta al puerto 2025:** revisa que la terminal del servidor esté abierta, dentro de mi-agente y sin errores de carga.
- **No aparece agente_christopher:** verifica el listado y que ejecutaste el servidor de esta copia.
- **Clave rechazada, modelo inexistente o cuota agotada:** revisa la configuración/cuenta del proveedor en la terminal; no pegues claves en celdas ni capturas.
- **Timeout:** revisa la terminal antes de reintentar: la tarea anterior podría continuar. TIMEOUT está en la primera celda.
- **Imports ausentes:** selecciona el kernel .venv correcto; las dependencias ya se instalaron en ese entorno.
- **Fallo de memoria:** ejecuta las dos llamadas sin crear otro thread entre ellas.
- **Reinicio de kernel:** se pierden las variables locales; vuelve a ejecutar desde la configuración. Las salidas guardadas siguen en el archivo hasta que las reemplaces o limpies.